In [2]:
# Bloque 1 — Extraccion de bytes crudos de paquetes (D-PACK, Hwang 2020 Escenario 3)
# El paper trabaja con los primeros n paquetes x l bytes por flujo como input de la CNN.
# n=2, l=80  ->  INPUT_DIM = 160 bytes por flujo.
import warnings
warnings.filterwarnings('ignore')

import os
import subprocess
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

import dpkt

# Hiperparametros de muestreo D-PACK (Hwang 2020, Tabla 13: mejor config n=2, l=80)
N_PKTS    = 2    # primeros n paquetes por flujo
L_BYTES   = 80   # primeros l bytes por paquete
INPUT_DIM = N_PKTS * L_BYTES  # = 160

RANDOM_SEED   = 42
LEARNING_RATE = 0.001
BATCH_SIZE    = 256
EPOCHS_CNN    = 30
EPOCHS_AE     = 30
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

EDITCAP   = r'C:\Program Files\Wireshark\editcap.exe'
DATA_PATH = str(Path.home() / ".cache" / "kagglehub" / "datasets" / "nguyenquanitmo" / "mirai-raw-dataset" / "versions" / "2")
TEMP_DIR  = str(Path.cwd().parent / "data" / "temp_pcap")
os.makedirs(TEMP_DIR, exist_ok=True)

# Registro de ficheros (igual que Markov Mirai -- Hwang Tabla 12: 9 clases)
FILE_REGISTRY = []
for i in range(1, 31):
    FILE_REGISTRY.append((f'non-attack{i}.pcapng', 'Mirai_CnC', 1))
for i in range(1, 31):
    if i == 16:
        continue
    FILE_REGISTRY.append((f'non-botnet{i}.pcapng', 'Normal', 0))
ATTACK_FILES = [
    ('ack_attack',   'ACK_Flood',   1),
    ('dns_attack',   'DNS_Flood',   1),
    ('greip_attack', 'GREIP_Flood', 1),
    ('http_attack',  'HTTP_Flood',  1),
    ('syn_attack',   'SYN_Flood',   1),
    ('udp_attack',   'UDP_Flood',   1),
    ('vse_attack',   'VSE_Flood',   1),
]
for base, class_name, label in ATTACK_FILES:
    for i in range(1, 4):
        FILE_REGISTRY.append((f'{base}{i}.pcapng', class_name, label))


def convert_to_pcap(pcapng_path):
    basename  = os.path.splitext(os.path.basename(pcapng_path))[0]
    pcap_path = os.path.join(TEMP_DIR, f'{basename}.pcap')
    subprocess.run([EDITCAP, '-F', 'pcap', pcapng_path, pcap_path],
                   capture_output=True, timeout=120, check=True)
    return pcap_path


def extract_flow_bytes(pcap_path, class_name, label, n=N_PKTS, l=L_BYTES):
    """Agrupa paquetes en flujos por 5-tupla y extrae los primeros n*l bytes."""
    flow_pkts = defaultdict(list)
    with open(pcap_path, 'rb') as f:
        reader = dpkt.pcap.Reader(f)
        for ts, raw in reader:
            try:
                eth = dpkt.ethernet.Ethernet(raw)
                if not isinstance(eth.data, dpkt.ip.IP):
                    continue
                ip    = eth.data
                proto = ip.p
                if isinstance(ip.data, (dpkt.tcp.TCP, dpkt.udp.UDP)):
                    sport = ip.data.sport
                    dport = ip.data.dport
                else:
                    sport = dport = 0
                fwd = (ip.src, ip.dst, sport, dport, proto)
                # 5-tupla unidireccional (paper sec. III-A)
                flow_pkts[fwd].append((ts, raw))
            except Exception:
                continue

    records = []
    for key, pkts in flow_pkts.items():
        pkts.sort(key=lambda x: x[0])
        parts = []
        for _, raw in pkts[:n]:
            b = np.frombuffer(raw, dtype=np.uint8)[:l]
            if len(b) < l:
                b = np.concatenate([b, np.zeros(l - len(b), dtype=np.uint8)])
            parts.append(b.astype(np.float32) / 255.0)
        while len(parts) < n:
            parts.append(np.zeros(l, dtype=np.float32))
        records.append({
            'bytes':      np.concatenate(parts),   # shape (n*l,) = (160,)
            'class_name': class_name,
            'label':      label,
        })
    return records


all_records = []
for i, (filename, class_name, label) in enumerate(FILE_REGISTRY):
    pcapng_path = os.path.join(DATA_PATH, filename)
    print(f'[{i+1}/{len(FILE_REGISTRY)}] {filename}  ({class_name})')
    try:
        pcap_path = convert_to_pcap(pcapng_path)
        records   = extract_flow_bytes(pcap_path, class_name, label)
        os.remove(pcap_path)
        all_records.extend(records)
        print(f'    {len(records):,} flujos')
    except Exception as e:
        print(f'    ERROR: {e}')

class_names_all = np.array([r['class_name'] for r in all_records])
labels_all      = np.array([r['label']      for r in all_records], dtype=np.int32)
X_all           = np.stack([r['bytes']      for r in all_records]).astype(np.float32)

print(f'\nTotal flujos: {len(X_all):,}  |  INPUT_DIM: {X_all.shape[1]}')
for cn in sorted(set(class_names_all)):
    print(f'  {cn:<14}: {(class_names_all==cn).sum():>7,}')

[1/80] non-attack1.pcapng  (Mirai_CnC)
    1,547 flujos
[2/80] non-attack2.pcapng  (Mirai_CnC)
    1,383 flujos
[3/80] non-attack3.pcapng  (Mirai_CnC)
    1,423 flujos
[4/80] non-attack4.pcapng  (Mirai_CnC)
    1,420 flujos
[5/80] non-attack5.pcapng  (Mirai_CnC)
    1,727 flujos
[6/80] non-attack6.pcapng  (Mirai_CnC)
    1,721 flujos
[7/80] non-attack7.pcapng  (Mirai_CnC)
    1,683 flujos
[8/80] non-attack8.pcapng  (Mirai_CnC)
    1,411 flujos
[9/80] non-attack9.pcapng  (Mirai_CnC)
    1,643 flujos
[10/80] non-attack10.pcapng  (Mirai_CnC)
    1,663 flujos
[11/80] non-attack11.pcapng  (Mirai_CnC)
    1,665 flujos
[12/80] non-attack12.pcapng  (Mirai_CnC)
    1,677 flujos
[13/80] non-attack13.pcapng  (Mirai_CnC)
    1,540 flujos
[14/80] non-attack14.pcapng  (Mirai_CnC)
    1,594 flujos
[15/80] non-attack15.pcapng  (Mirai_CnC)
    1,597 flujos
[16/80] non-attack16.pcapng  (Mirai_CnC)
    1,817 flujos
[17/80] non-attack17.pcapng  (Mirai_CnC)
    1,777 flujos
[18/80] non-attack18.pcapng  (Mi

In [3]:
# Bloque 2 — Subsampling (Hwang Tabla 12) y split train/test

HWANG_CLASS_TOTAL = {
    'Normal':       68200 + 8525,
    'ACK_Flood':     6600 +  825,
    'DNS_Flood':     4312 +  539,
    'Mirai_CnC':    68200 + 8525,
    'GREIP_Flood':  24712 + 3089,
    'HTTP_Flood':     120 +   15,
    'SYN_Flood':    68200 + 8525,
    'UDP_Flood':    28816 + 3062,
    'VSE_Flood':     4432 +  554,
}

np.random.seed(42)
idx_keep_list = []
for cls_name, n_total in HWANG_CLASS_TOTAL.items():
    idx_cls = np.where(class_names_all == cls_name)[0]
    if len(idx_cls) == 0:
        print(f'  AVISO: clase {cls_name!r} no encontrada')
        continue
    available = len(idx_cls)
    if available > n_total:
        idx_cls = np.random.choice(idx_cls, n_total, replace=False)
    idx_keep_list.append(idx_cls)
    print(f'  {cls_name:<14}: disponible={available:>7,}  usado={len(idx_cls):>7,}')

idx_keep = np.sort(np.concatenate(idx_keep_list))
X        = X_all[idx_keep]
y_class  = class_names_all[idx_keep]
y_bin    = labels_all[idx_keep]

# Hwang Escenario 3: CNN entrena con TODAS las clases (benigno + maligno),
# porque el dataset solo tiene 1 tipo benigno (Normal).
# La AE se entrena unicamente con trafico Normal.
class_enc = LabelEncoder()
y_multi   = class_enc.fit_transform(y_class)  # 9 clases: Normal + 8 ataques
N_CLASSES = len(class_enc.classes_)
print(f'\nClases CNN ({N_CLASSES}): {list(class_enc.classes_)}')

# Split 80/20 estratificado por clase multi-clase
X_train, X_test, y_train_multi, y_test_multi, y_train_bin, y_test_bin = train_test_split(
    X, y_multi, y_bin, test_size=1/9, random_state=RANDOM_SEED, stratify=y_multi  # Hwang Tabla 12: ratio 8:1 (88.89/11.11)
)

# Los bytes ya estan normalizados a [0,1] en la extraccion (/255).
# No se aplica MinMaxScaler adicional.
X_train_cnn     = X_train
X_train_autoenc = X_train[y_train_bin == 0]  # solo trafico Normal para AE

print(f'Train: {len(X_train):,} | Test: {len(X_test):,}')
print(f'AE Train (Normal): {len(X_train_autoenc):,}')
print(f'INPUT_DIM: {INPUT_DIM} | N_CLASSES: {N_CLASSES}')
print(f'Normal(0): {(y_bin==0).sum():,}  |  Ataque(1): {(y_bin==1).sum():,}')

  Normal        : disponible=  8,091  usado=  8,091
  ACK_Flood     : disponible=167,001  usado=  7,425
  DNS_Flood     : disponible=309,547  usado=  4,851
  Mirai_CnC     : disponible= 49,547  usado= 49,547
  GREIP_Flood   : disponible=  4,973  usado=  4,973
  HTTP_Flood    : disponible=  5,348  usado=    135
  SYN_Flood     : disponible=207,156  usado= 76,725
  UDP_Flood     : disponible=155,183  usado= 31,878
  VSE_Flood     : disponible=210,172  usado=  4,986

Clases CNN (9): ['ACK_Flood', 'DNS_Flood', 'GREIP_Flood', 'HTTP_Flood', 'Mirai_CnC', 'Normal', 'SYN_Flood', 'UDP_Flood', 'VSE_Flood']
Train: 167,654 | Test: 20,957
AE Train (Normal): 7,192
INPUT_DIM: 160 | N_CLASSES: 9
Normal(0): 8,091  |  Ataque(1): 180,520


In [4]:
# Bloque 3 — Definicion y Entrenamiento del modelo D-PACK

class DPACK(nn.Module):
    def __init__(self, input_dim, n_classes):
        super(DPACK, self).__init__()
        self.input_dim = input_dim
        self.conv1 = nn.Sequential(nn.Conv1d(1, 32, kernel_size=6, stride=1, padding=5), nn.ReLU(), nn.BatchNorm1d(32))
        self.pool1 = nn.MaxPool1d(kernel_size=2, stride=2)
        self.conv2 = nn.Sequential(nn.Conv1d(32, 64, kernel_size=6, stride=1, padding=5), nn.ReLU(), nn.BatchNorm1d(64))
        self.pool2 = nn.MaxPool1d(kernel_size=2, stride=2)
        self._conv_out_size = self._get_conv_output_size()
        self.fc5 = nn.Sequential(nn.Linear(self._conv_out_size, 1024), nn.ReLU(), nn.BatchNorm1d(1024))
        self.fc6 = nn.Sequential(nn.Linear(1024, 25), nn.ReLU(), nn.BatchNorm1d(25))
        self.fc7 = nn.Linear(25, n_classes)
        self.ae_fc8  = nn.Sequential(nn.Linear(1024, 512), nn.ReLU())
        self.ae_fc9  = nn.Sequential(nn.Linear(512,  256), nn.ReLU())
        self.ae_fc10 = nn.Sequential(nn.Linear(256,  512), nn.ReLU())
        self.ae_fc11 = nn.Sequential(nn.Linear(512, 1024), nn.ReLU())
        self.ae_out  = nn.Linear(1024, input_dim)

    def _get_conv_output_size(self):
        with torch.no_grad():
            dummy = torch.zeros(1, 1, self.input_dim)
            out   = self.pool2(self.conv2(self.pool1(self.conv1(dummy))))
            return int(out.numel())

    def forward(self, x):
        h = x.unsqueeze(1)
        h = self.pool1(self.conv1(h))
        h = self.pool2(self.conv2(h))
        h = h.view(h.size(0), -1)
        h5      = self.fc5(h)
        cnn_out = self.fc7(self.fc6(h5))
        ae_out  = self.ae_out(self.ae_fc11(self.ae_fc10(self.ae_fc9(self.ae_fc8(h5)))))
        return cnn_out, ae_out, h5


def train_cnn_phase(model, X_tr, y_tr, epochs, batch_size, lr, device):
    model.train()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    loader    = DataLoader(TensorDataset(torch.FloatTensor(X_tr), torch.LongTensor(y_tr)),
                           batch_size=batch_size, shuffle=True, num_workers=0)
    for epoch in range(epochs):
        total_loss, correct, total = 0.0, 0, 0
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            cnn_out, ae_out, _ = model(X_batch)
            loss = criterion(cnn_out, y_batch) + nn.MSELoss()(ae_out, X_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            correct    += (cnn_out.argmax(dim=1) == y_batch).sum().item()
            total      += y_batch.size(0)
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f'  Epoca {epoch+1:>3}/{epochs} | Loss: {total_loss/len(loader):.4f} | Acc: {100*correct/total:.2f}%')
    return model


def train_autoencoder_phase(model, X_ae, epochs, batch_size, lr, device):
    model.train()
    ae_params = (list(model.fc5.parameters())    + list(model.ae_fc8.parameters()) +
                 list(model.ae_fc9.parameters()) + list(model.ae_fc10.parameters()) +
                 list(model.ae_fc11.parameters()) + list(model.ae_out.parameters()))
    optimizer = optim.Adam(ae_params, lr=lr)
    criterion = nn.MSELoss()
    loader    = DataLoader(TensorDataset(torch.FloatTensor(X_ae)),
                           batch_size=batch_size, shuffle=True, num_workers=0)
    for epoch in range(epochs):
        total_mse = 0.0
        for (X_batch,) in loader:
            X_batch = X_batch.to(device)
            optimizer.zero_grad()
            _, ae_out, _ = model(X_batch)
            loss = criterion(ae_out, X_batch)
            loss.backward()
            optimizer.step()
            total_mse += loss.item()
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f'  Epoca {epoch+1:>3}/{epochs} | MSE: {total_mse/len(loader):.6f}')
    return model


def compute_threshold(model, X_benign, device, batch_size=512):
    model.eval()
    mse_list = []
    with torch.no_grad():
        for i in range(0, len(X_benign), batch_size):
            X_batch = torch.FloatTensor(X_benign[i:i+batch_size]).to(device)
            _, ae_out, _ = model(X_batch)
            mse_list.extend(((ae_out - X_batch) ** 2).mean(dim=1).cpu().numpy())
    mse_arr = np.array(mse_list)
    max_mse, p99, std = np.max(mse_arr), np.percentile(mse_arr, 99), np.std(mse_arr)
    threshold = p99 if (max_mse - p99) > 3 * std else max_mse
    print(f'  Umbral = {threshold:.6f} ({"percentil 99" if threshold == p99 else "max MSE"})')
    return threshold


torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

model = DPACK(input_dim=INPUT_DIM, n_classes=N_CLASSES).to(device)
print(f'Parametros totales: {sum(p.numel() for p in model.parameters()):,}')

print('\nFASE 1 - CNN (entrena con todas las clases)')
model = train_cnn_phase(model, X_train_cnn, y_train_multi, EPOCHS_CNN, BATCH_SIZE, LEARNING_RATE, device)

print('\nFASE 2 - Autoencoder (entrena solo con trafico Normal)')
model = train_autoencoder_phase(model, X_train_autoenc, EPOCHS_AE, BATCH_SIZE, LEARNING_RATE, device)

print('\nUMBRAL')
threshold = compute_threshold(model, X_train_autoenc, device)

Parametros totales: 4,336,821

FASE 1 - CNN (entrena con todas las clases)
  Epoca   1/30 | Loss: 0.2483 | Acc: 95.43%
  Epoca   5/30 | Loss: 0.0542 | Acc: 98.40%
  Epoca  10/30 | Loss: 0.0300 | Acc: 99.07%
  Epoca  15/30 | Loss: 0.0168 | Acc: 99.51%
  Epoca  20/30 | Loss: 0.0119 | Acc: 99.68%
  Epoca  25/30 | Loss: 0.0083 | Acc: 99.79%
  Epoca  30/30 | Loss: 0.0074 | Acc: 99.81%

FASE 2 - Autoencoder (entrena solo con trafico Normal)
  Epoca   1/30 | MSE: 0.005146
  Epoca   5/30 | MSE: 0.001104
  Epoca  10/30 | MSE: 0.000734
  Epoca  15/30 | MSE: 0.000609
  Epoca  20/30 | MSE: 0.000485
  Epoca  25/30 | MSE: 0.000406
  Epoca  30/30 | MSE: 0.000340

UMBRAL
  Umbral = 0.003565 (percentil 99)


In [5]:
# Bloque 4 — Resultados

def predict(model, X, threshold, device, batch_size=512):
    model.eval()
    mse_list = []
    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            X_batch = torch.FloatTensor(X[i:i+batch_size]).to(device)
            _, ae_out, _ = model(X_batch)
            mse_list.extend(((ae_out - X_batch) ** 2).mean(dim=1).cpu().numpy())
    return (np.array(mse_list) > threshold).astype(int)


def compute_metrics(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    accuracy  = (tp + tn) / (tp + tn + fp + fn)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    far       = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    fnr       = fn / (tp + fn) if (tp + fn) > 0 else 0.0
    return {'Accuracy': accuracy, 'Precision': precision, 'Recall': recall,
            'F1': f1, 'FAR': far, 'FNR': fnr, 'TP': tp, 'TN': tn, 'FP': fp, 'FN': fn}


y_pred  = predict(model, X_test, threshold, device)
metrics = compute_metrics(y_test_bin, y_pred)

SEP = '=' * 50
print('D-PACK - Dataset Mirai (Hwang 2020, Escenario 3)')
print(SEP)
for k in ('Accuracy', 'Precision', 'Recall', 'F1', 'FAR', 'FNR'):
    print(f'  {k:<12}: {metrics[k]*100:.2f}%')
print(SEP)
print(f'  TP={int(metrics["TP"]):,}  TN={int(metrics["TN"]):,}  FP={int(metrics["FP"]):,}  FN={int(metrics["FN"]):,}')
print()
print('Referencia paper (Tabla 14, n=2, l=80):')
print('  Accuracy ~99%  Precision ~99%  Recall ~99%  F1 ~99%  FNR <1%')

D-PACK - Dataset Mirai (Hwang 2020, Escenario 3)
  Accuracy    : 99.45%
  Precision   : 99.58%
  Recall      : 99.85%
  F1          : 99.71%
  FAR         : 9.45%
  FNR         : 0.15%
  TP=20,028  TN=814  FP=85  FN=30

Referencia paper (Tabla 14, n=2, l=80):
  Accuracy ~99%  Precision ~99%  Recall ~99%  F1 ~99%  FNR <1%
